In [1]:
!pip install tensorflow opencv-python matplotlib ipywidgets
Python

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)


NameError: name 'Python' is not defined

In [2]:
# Imports

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout,
    Flatten, BatchNormalization, Input
)
from collections import deque
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
from pathlib import Path

print("OpenCV version:", cv2.__version__)

OpenCV version: 4.13.0


In [3]:
# Where is your trained weights file?
WEIGHTS_PATH = r"C:\Users\Lfellous\Desktop\FER2013_results\emotion_model.weights.h5"

# Folder where you can put test images (optional)
TEST_IMAGES_FOLDER = r"C:\Users\Lfellous\Desktop\FER2013\test_faces"

emotion_dict = {
    0: "Angry",     1: "Disgusted",  2: "Fearful",
    3: "Happy",     4: "Neutral",    5: "Sad",     6: "Surprised"
}

EMOTION_COLORS = {
    0: (0, 0, 255),     1: (0, 128, 0),      2: (255, 0, 255),
    3: (0, 255, 255),   4: (255, 255, 0),    5: (255, 0, 0),
    6: (0, 165, 255)
}

print("Expecting model weights at:", WEIGHTS_PATH)

Expecting model weights at: C:\Users\Lfellous\Desktop\FER2013_results\emotion_model.weights.h5


In [4]:
# Rebuild model & load trained weights
# Must match the exact architecture from model_training.ipynb

model = Sequential([
    Input(shape=(48, 48, 1)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2), Dropout(0.25),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2), Dropout(0.25),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2), Dropout(0.25),

    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2), Dropout(0.25),

    Flatten(),
    Dense(512, activation='relu'), BatchNormalization(), Dropout(0.4),
    Dense(256, activation='relu'), Dropout(0.3),
    Dense(7, activation='softmax')
])

model.load_weights(WEIGHTS_PATH)
print("✅ Model architecture rebuilt and weights loaded")
model.summary()

✅ Model architecture rebuilt and weights loaded


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 6, 6, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,899,751 (7.25 MB)

 Trainable params: 1,897,319 (7.24 MB)

 Non-trainable params: 2,432 (9.50 KB)

In [5]:
# Face detector + prediction logic

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Temporal smoothing (helps reduce label flickering)
prediction_buffer = deque(maxlen=8)

def predict_emotion(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5, minSize=(30,30))

    result = frame.copy()

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_resized = cv2.resize(roi_gray, (48, 48))
        roi_input = roi_resized.astype('float32') / 255.0
        roi_input = np.expand_dims(np.expand_dims(roi_input, -1), 0)

        # Predict
        raw_pred = model.predict(roi_input, verbose=0)[0]

        # Smooth predictions over time
        prediction_buffer.append(raw_pred)
        smoothed = np.mean(prediction_buffer, axis=0)
        emotion_idx = int(np.argmax(smoothed))
        confidence = smoothed[emotion_idx] * 100

        color = EMOTION_COLORS[emotion_idx]
        label = f"{emotion_dict[emotion_idx]}  {confidence:.0f}%"

        # Draw rectangle + label
        cv2.rectangle(result, (x, y), (x+w, y+h), color, 2)
        cv2.putText(result, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    return result, len(faces)

In [8]:
# Option A: Real-time webcam demo (OpenCV)


def webcam_demo():
    cap = cv2.VideoCapture(0)  # 0 = default webcam
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    prediction_buffer.clear()
    print("Webcam opened. Press 'q' to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        result, n_faces = predict_emotion(frame)

        # Show number of faces detected
        cv2.putText(result, f"Faces: {n_faces}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

        cv2.imshow("Emotion Recognition - press q to quit", result)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam session closed.")

# Uncomment to run:
webcam_demo()

Webcam opened. Press 'q' to quit.
Webcam session closed.


In [7]:
# Option B: Test on a single image file


def predict_single_image(image_path):
    if not os.path.exists(image_path):
        print(f"File not found: {image_path}")
        return

    frame = cv2.imread(image_path)
    if frame is None:
        print("Could not read image.")
        return

    prediction_buffer.clear()
    result, n_faces = predict_emotion(frame)

    print(f"Image: {Path(image_path).name}  —  {n_faces} face(s) detected")

    # Show result with matplotlib (better quality in notebook)
    rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(rgb)
    plt.axis('off')
    plt.title(Path(image_path).name)
    plt.show()

# Example usage – change the path
# predict_single_image(r"C:\Users\YourName\Pictures\test_faces\happy_person.jpg")